In [121]:
import duckdb

In [122]:
with duckdb.connect('resident_calibration.duckdb') as con:
    # persons = con.table("calibration_staging.stg_abm3_persons")
    # mgra_data = con.read_csv('local_data/mgra15_based_input2022.csv')
    
    # result = persons.join(mgra_data, condition='stg_abm3_persons.workplace_zone_id = local_data/mgra15_based_input2022.mgra').df()
    # # # Or specify columns explicitly:
    # # result = persons.join(mgra_data, condition='persons.mgra = mgra_data.mgra').df()

    result = con.execute("""
        SELECT ptype, is_student,is_worker,free_parking_at_work, m.exp_daily as workloc_parkcost, 1 as finalweights
        FROM calibration_staging.stg_abm3_persons p
        LEFT JOIN read_csv('local_data/mgra15_based_input2022.csv') m
        ON p.workplace_zone_id = m.mgra
    """).fetchdf()

    #result_v2 = result.query('workloc_parkcost > 0').groupby('ptype').finalweights.sum()

In [123]:
result.query('workloc_parkcost > 0 and (is_worker == True or is_student == True)')#.groupby('ptype').finalweights.sum()

,ptype,is_student,is_worker,free_parking_at_work,workloc_parkcost,finalweights
0,1,False,True,True,3.572,1
10,1,False,True,False,0.407,1
20,1,False,True,True,6.093,1
25,1,False,True,True,8.918,1
43,1,False,True,False,3.895,1
...,...,...,...,...,...,...
3165653,2,False,True,False,2.281,1
3165654,1,False,True,True,16.189,1
3165656,1,False,True,False,2.297,1
3165659,1,False,True,True,8.172,1


In [129]:
# Create the aggregation
ptype_sums = result.query('workloc_parkcost > 0').groupby('ptype').finalweights.sum()

# Map it back to the original dataframe
result['ptype_weight_sum'] = result['ptype'].map(ptype_sums)
result['free_parking_weight'] = result.free_parking_at_work * result.finalweights / result.ptype_weight_sum


In [125]:
result

,ptype,is_student,is_worker,free_parking_at_work,workloc_parkcost,finalweights,ptype_weight_sum,free_parking_weight,final_fp_weight
0,1,False,True,True,3.572,1,267169.0,0.000004,2.183700
1,1,False,True,False,0.000,1,267169.0,0.000000,2.183700
2,1,False,True,True,0.000,1,267169.0,0.000004,2.183700
3,1,False,True,True,0.000,1,267169.0,0.000004,2.183700
4,1,False,True,True,0.000,1,267169.0,0.000004,2.183700
...,...,...,...,...,...,...,...,...,...
3166653,6,True,False,False,NaN,1,2647.0,0.000000,2.219494
3166654,7,True,False,False,NaN,1,NaN,NaN,0.000000
3166655,7,True,False,False,NaN,1,NaN,NaN,0.000000
3166656,8,True,False,False,NaN,1,NaN,NaN,0.000000


In [126]:
ptype_sums.to_dict()

{1: 267169, 2: 50698, 3: 17337, 6: 2647}

In [127]:
result.query('workloc_parkcost > 0 and (is_worker == True or is_student == True)').groupby('ptype').free_parking_weight.sum()

ptype
1    0.530664
2    0.506923
3    0.476092
6    0.499811
Name: free_parking_weight, dtype: float64

In [104]:
result.query('free_parking_at_work == True').ptype.value_counts() / result.query('workloc_parkcost > 0').ptype.value_counts()

ptype
1    2.183700
2    2.308848
3    1.761897
6    2.219494
Name: count, dtype: float64

In [89]:
result.groupby('ptype').free_parking_weight.sum()

ptype
1    2.183700
2    2.308848
3    1.761897
4    0.000000
5    0.000000
6    2.219494
7    0.000000
8    0.000000
Name: free_parking_weight, dtype: float64

In [0]:
with duckdb.connect('files.duckdb') as con:
    con.sql("SELECT * FROM stg_trips ORDER BY trip_id ASC").show()
